# 2. praktika: datu-multzoa aztertu eta prestatu

**Helburua**: entrenamendu-datuak ondo ezagutu (banaketa, formatua, luzera, kalitatea), txat-txantiloia ulertu eta adibide berriak gehitu.

Saio honetan ez da GPUrik behar (tokenizatzailea bakarrik kargatuko dugu).

## 1. Ingurunea

In [ ]:
# COLAB: liburutegiak instalatu (bertsio finkoak, unitate osoan berdinak)
!pip install -q transformers==5.22.2 peft==0.21.2 trl==1.17.1 datasets==5.2.0

In [ ]:
# COLAB: Google Drive muntatu (emaitzak gordetzeko eta datuak irakurtzeko)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
# Aldatu zure Drive-ko karpetara. Bertan egon behar dute txartelak_train.jsonl eta txartelak_test.jsonl
KARPETA = "/content/drive/MyDrive/llm-egokitzapena"
os.makedirs(KARPETA, exist_ok=True)
TRAIN_FITX = f"{KARPETA}/txartelak_train.jsonl"
TEST_FITX = f"{KARPETA}/txartelak_test.jsonl"

# Fitxategiak Drive-n ez badaude, igo eskuz
if not os.path.exists(TRAIN_FITX):
    from google.colab import files  # COLAB
    igoak = files.upload()
    for izena in igoak:
        os.replace(izena, f"{KARPETA}/{izena}")
print(os.listdir(KARPETA))

In [ ]:
import json
def irakurri_jsonl(fitx):
    with open(fitx, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_lista = irakurri_jsonl(TRAIN_FITX)
test_lista = irakurri_jsonl(TEST_FITX)
SYSTEM = train_lista[0]["prompt"][0]["content"]   # sistema-mezua datuetatik (beti berdina!)
print(len(train_lista), len(test_lista))
print(SYSTEM)

## 2. Datuak pandas-era

In [ ]:
import pandas as pd

def taulara(lista):
    lerroak = []
    for a in lista:
        irteera = json.loads(a["completion"][0]["content"])
        lerroak.append({"testua": a["prompt"][1]["content"], **irteera, "mota": a.get("mota", "")})
    return pd.DataFrame(lerroak)

df_train = taulara(train_lista)
df_test = taulara(test_lista)
df_train.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
df_train["kategoria"].value_counts().plot.bar(ax=ax[0], title="Kategoria")
df_train["lehentasuna"].value_counts().plot.bar(ax=ax[1], title="Lehentasuna")
df_train["kokapena"].notna().map({True: "bai", False: "ez"}).value_counts().plot.bar(ax=ax[2], title="Kokapena du?")
plt.tight_layout(); plt.show()

In [ ]:
# Zein kokapen agertzen dira train-en eta test-en?
print("TRAIN:", sorted(df_train["kokapena"].dropna().unique()))
print("TEST: ", sorted(df_test["kokapena"].dropna().unique()))
print("Test-en bakarrik:", set(df_test["kokapena"].dropna()) - set(df_train["kokapena"].dropna()))

In [ ]:
# Test multzoa: ikusiak vs berriak
df_test["mota"].value_counts()

## 3. Txat-txantiloia

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

adib = train_lista[0]
print("=== ENTRENAMENDUAN (elkarrizketa osoa) ===")
print(tokenizer.apply_chat_template(adib["prompt"] + adib["completion"], tokenize=False))
print("=== INFERENTZIAN (erantzunik gabe) ===")
print(tokenizer.apply_chat_template(adib["prompt"], tokenize=False, add_generation_prompt=True))

In [ ]:
# Token bereziak: bakoitza token BAKARRA da
for t in ["<|im_start|>", "<|im_end|>", "assistant", "inprimagailua", "B14 gelako"]:
    ids = tokenizer.encode(t, add_special_tokens=False)
    print(f"{t!r:20} -> {ids}  {[tokenizer.decode([i]) for i in ids]}")
print("EOS:", tokenizer.eos_token, "| PAD:", tokenizer.pad_token)

> **Galdera**: zenbat token behar ditu euskarazko hitz batek ingelesezko batek baino? Probatu zuk zeuk hitz batzuekin. Zer ondorio du horrek eredu txiki batentzat?

## 4. Luzera tokenetan

In [ ]:
luzerak = [len(tokenizer.apply_chat_template(a["prompt"] + a["completion"])["input_ids"]) for a in train_lista]
sys_luz = len(tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM}])["input_ids"])
erant_luz = [len(tokenizer.encode(a["completion"][0]["content"])) for a in train_lista]

print(f"Adibide osoa: min {min(luzerak)}, max {max(luzerak)}, batez beste {sum(luzerak)/len(luzerak):.0f}")
print(f"Sistema-mezua: {sys_luz} token ({sys_luz / (sum(luzerak)/len(luzerak)):.0%})")
print(f"Erantzuna (galera hemen kalkulatzen da): batez beste {sum(erant_luz)/len(erant_luz):.0f} token")
plt.hist(luzerak, bins=30); plt.xlabel("tokenak"); plt.title("Adibideen luzera"); plt.show()

## 5. Kalitate-egiaztapenak

In [ ]:
KATEGORIAK = {"sarea", "hardwarea", "softwarea", "kontuak", "inprimagailua"}
LEHENTASUNAK = {"baxua", "ertaina", "altua"}

def egiaztatu_adibidea(a):
    """Adibide baten arazoen zerrenda itzultzen du (hutsik = ondo)."""
    arazoak = []
    if set(a) - {"prompt", "completion", "mota"}:
        arazoak.append(f"gako arraroak: {set(a)}")
    rolak = [m["role"] for m in a["prompt"]] + [m["role"] for m in a["completion"]]
    if rolak != ["system", "user", "assistant"]:
        arazoak.append(f"rolak: {rolak}")
    if a["prompt"][0]["content"] != SYSTEM:
        arazoak.append("sistema-mezu desberdina")
    try:
        j = json.loads(a["completion"][0]["content"])
    except json.JSONDecodeError:
        return arazoak + ["JSON baliogabea"]
    if list(j) != ["kategoria", "lehentasuna", "kokapena"]:
        arazoak.append(f"eremuak/ordena: {list(j)}")
    if j.get("kategoria") not in KATEGORIAK:
        arazoak.append(f"kategoria: {j.get('kategoria')}")
    if j.get("lehentasuna") not in LEHENTASUNAK:
        arazoak.append(f"lehentasuna: {j.get('lehentasuna')}")
    # kokapenaren erroa (azken letra gabe) testuan egon behar da: "liburutegia" -> "liburutegi"
    if j.get("kokapena") is not None and j["kokapena"].split()[0][:-1].lower() not in a["prompt"][1]["content"].lower():
        arazoak.append("kokapena ez dago testuan")
    return arazoak

for izena, lista in [("train", train_lista), ("test", test_lista)]:
    txarrak = [(i, egiaztatu_adibidea(a)) for i, a in enumerate(lista) if egiaztatu_adibidea(a)]
    print(izena, "->", len(txarrak), "adibide arazoekin", txarrak[:3])

In [ ]:
# Bikoiztuak eta datu-ihesa (test-eko testuak train-en?)
bikoiztuak = df_train["testua"].duplicated().sum()
ihesa = set(df_test["testua"]) & set(df_train["testua"])
print("Bikoiztuak train-en:", bikoiztuak)
print("Test-eko testuak train-en ere:", len(ihesa))
list(ihesa)[:5]

> **Galdera**: bikoiztuak edo ihesak aurkitu badituzu, zergatik gertatu da? (aholkua: txantiloiekin sortutako datuak). Zer egin beharko genuke?

In [ ]:
# Konpondu: bikoiztuak kendu eta test-ean dauden testuak train-etik atera
ikusiak, train_garbia = set(df_test["testua"]), []
for a in train_lista:
    t = a["prompt"][1]["content"]
    if t not in ikusiak:
        ikusiak.add(t); train_garbia.append(a)
print(len(train_lista), "->", len(train_garbia))

> **Oharra**: 3. saioan `txartelak_train.jsonl` originala erabiliko dugu denok emaitza berdinak lortzeko. Bertsio garbia `txartelak_train_garbia.jsonl` gisa gordeko dugu, 4. saioan alderatzeko (iterazioa!).

In [ ]:
def idatzi_jsonl(lista, fitx):
    with open(fitx, "w", encoding="utf-8") as f:
        for a in lista:
            f.write(json.dumps(a, ensure_ascii=False) + "\n")

idatzi_jsonl(train_garbia, f"{KARPETA}/txartelak_train_garbia.jsonl")

## 6. `datasets` liburutegia

In [ ]:
from datasets import load_dataset

train = load_dataset("json", data_files=TRAIN_FITX, split="train")
print(train)

kokapenarekin = train.filter(lambda a: '"kokapena": null' not in a["completion"][0]["content"])
print("Kokapenarekin:", len(kokapenarekin))

train_msg = train.map(lambda a: {"messages": a["prompt"] + a["completion"]},
                      remove_columns=["prompt", "completion"])
print(train_msg[0]["messages"][-1])

## 7. Zure adibideak

Idatzi eskuz **20 txartel berri**, irakasle batek benetan idatziko lukeen bezala: laburrak, luzeak, akatsekin, gaztelaniazko hitzekin, bi arazo batera... Saiatu txantiloietan **ez** dauden kasuak sortzen. Gero egiaztatu `egiaztatu_adibidea` funtzioarekin.

In [ ]:
def adibidea(testua, kategoria, lehentasuna, kokapena=None):
    erantzuna = json.dumps({"kategoria": kategoria, "lehentasuna": lehentasuna, "kokapena": kokapena},
                           ensure_ascii=False)
    return {"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": testua}],
            "completion": [{"role": "assistant", "content": erantzuna}]}

nireak = [
    adibidea("Aupa! A12ko proyektoreak ez du konektatzen HDMI kablearekin, gaur arratsaldean ikastaroa dut",
             "hardwarea", "altua", "A12 gela"),
    adibidea("moodlek errorea ematen dit ataza igotzerakoan, fitxategia handiegia omen da", "softwarea", "ertaina"),
    # ... ZURE 18 ADIBIDEAK HEMEN
]
for a in nireak:
    print(egiaztatu_adibidea(a) or "ondo", "|", a["prompt"][1]["content"][:60])

In [ ]:
idatzi_jsonl(nireak, f"{KARPETA}/txartelak_gehigarriak.jsonl")

## 8. Ariketak

1. Zein da lehentasun bakoitzaren banaketa kategoria bakoitzeko? (`pd.crosstab`). Orekatuta dago?
2. `sortu_datuak.py` script-a irakurri. Zein txantiloi daude `T_TEST`-en eta ez `T`-n? Zergatik da garrantzitsua hori?
3. **Datu sintetikoak LLM batekin**: *LLMak Python-etik* ikastaroko kodea erabiliz (Ollama edo API bat), sortu 30 txartel berri kategoria baterako. Berrikusi eta zuzendu eskuz. Zenbatek zuten akatsak?
4. **Proiektua**: aukeratu zuen ataza, diseinatu irteerako JSON-a eta sortu lehen 50 adibideak formatu honetan. Egokitu `egiaztatu_adibidea` zuen atazara.